In [12]:
import pandas as pd
import holidays

def load_daily_exits(path):
    df = pd.read_csv(
        path,
        names=["date", "hour", "origin", "dest", "trips"],
        usecols=["date", "dest", "trips"],          # skip columns we don't need
        dtype={"dest": "category", "trips": "int32"},
    )
    return df.groupby(["date", "dest"], observed=True)["trips"].sum().reset_index()

daily = pd.concat([
    load_daily_exits("../data/raw/date-hour-soo-dest-2024.csv.gz"),
    load_daily_exits("../data/raw/date-hour-soo-dest-2025.csv.gz"),
])
daily = daily.rename(columns={"dest": "station", "trips": "exits"})
daily["date"] = pd.to_datetime(daily["date"])
daily.head()

,date,station,exits
0,2024-01-01,12TH,1181
1,2024-01-01,16TH,1994
2,2024-01-01,19TH,1291
3,2024-01-01,24TH,1877
4,2024-01-01,ANTC,347


In [13]:
top = daily.groupby("station", observed=True)["exits"].sum().nlargest(8).index
daily = daily[daily["station"].isin(top)].copy()
daily["station"] = daily["station"].astype(str)
print(list(top))

['EMBR', 'MONT', 'POWL', 'CIVC', 'DBRK', '16TH', '24TH', '12TH']


In [14]:
weather = pd.read_csv("../data/raw/weather.csv", parse_dates=["date"])
df = daily.merge(weather, on="date", how="left")
df.isna().sum()   # should be all zeros

date          0
station       0
exits         0
temp_max_f    0
precip_in     0
dtype: int64

In [15]:
us_holidays = holidays.US(years=[2024, 2025])

df["day_of_week"] = df["date"].dt.day_name()
df["month"] = df["date"].dt.month
df["is_holiday"] = df["date"].dt.date.isin(us_holidays).astype(int)

df.sample(10)

,date,station,exits,temp_max_f,precip_in,day_of_week,month,is_holiday
158,2024-01-20,MONT,4151,59.3,0.386,Saturday,1,0
5427,2025-11-09,CIVC,4431,76.9,0.000,Sunday,11,0
986,2024-05-03,24TH,5469,73.0,0.000,Friday,5,0
5189,2025-10-10,EMBR,15029,70.2,0.000,Friday,10,0
3779,2025-04-17,CIVC,9901,64.0,0.063,Thursday,4,0
876,2024-04-19,DBRK,5604,63.2,0.000,Friday,4,0
4199,2025-06-08,POWL,6622,64.8,0.000,Sunday,6,0
2459,2024-11-03,CIVC,3605,70.9,0.000,Sunday,11,0
3889,2025-05-01,16TH,6571,65.0,0.000,Thursday,5,0
2834,2024-12-20,24TH,5255,57.2,0.000,Friday,12,0


In [16]:
df.to_csv("../data/processed/bart_daily.csv", index=False)
print(df.shape)

(5848, 8)


In [17]:
import sys
sys.path.append("../src")

import pandas as pd
from linreg.preprocessing import split_by_date

df = pd.read_csv("../data/processed/bart_daily.csv")
train, test = split_by_date(df)

print(train['date'].max(), test['date'].min())
print(len(train) + len(test) == len(df))


2024-12-31 00:00:00 2025-01-01 00:00:00
True


In [18]:
train.head()


,date,station,exits,temp_max_f,precip_in,day_of_week,month,is_holiday
0,2024-01-01,12TH,1181,60.3,0.0,Monday,1,1
1,2024-01-01,16TH,1994,60.3,0.0,Monday,1,1
2,2024-01-01,24TH,1877,60.3,0.0,Monday,1,1
3,2024-01-01,CIVC,1944,60.3,0.0,Monday,1,1
4,2024-01-01,DBRK,1492,60.3,0.0,Monday,1,1
